# Capstone: which stale pages should we review first?
Lane: **Refresh / Content Opportunity Scoring**. This notebook mirrors the deployed paper: https://karyllegt.github.io/ml-starter-pack-flyrank-ai/

Run it top to bottom (Runtime / Run all). Every number in the paper comes from these cells.

## 1. Question
Which pages in a content library should a person review first because they appear to be declining? A page is labeled *declining* when its recent search trend is down. A good ranking puts declining pages near the top.

## 2. Data
The anonymized starter slice shipped with the FlyRank ML Internship repo: 30,000 pages, pseudonymized IDs, no client names, domains, URLs, titles, or keywords. Impressions use a 90-day window. The full hosted release was not used in this version.

In [ ]:
import os, sys, subprocess
import numpy as np, pandas as pd

DATA = "data/raw/content_refresh_anonymized.csv"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"

# Colab: get the repo (and the starter CSV) first
if "google.colab" in sys.modules and not os.path.exists(DATA):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "starter"], check=True)
    os.chdir("starter")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Codespaces / local: walk up from this notebook's folder until we find the repo root
root = os.getcwd()
while not os.path.exists(os.path.join(root, DATA)):
    parent = os.path.dirname(root)
    if parent == root:
        raise FileNotFoundError("Could not find " + DATA)
    root = parent
os.chdir(root)

df = pd.read_csv(DATA)
print(df.shape[0], "pages,", df.shape[1], "columns")

## 3. Methodology
- **Label:** declining = trend direction is "down".
- **Baseline (hand rule):** score = 90-day impressions if the page is stale (>= 180 days since update) AND visible (>= 500 impressions); otherwise 0.
- **Model:** decision tree, depth 3, balanced classes, six inputs: content age, days since update, impressions, average position, CTR, word count.
- **Validation:** hold out 20% of *clients* (not rows) as an unseen test set. The metric is Precision@K.
- **Leakage:** trend direction and trend percentage are never used as inputs.

In [ ]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y = df["is_declining_label"].values
print("Declining rate overall:", round(y.mean(), 3))

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GroupShuffleSplit

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

# Hold back 20% of CLIENTS as an unseen test set
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, y, groups=df["client_id"]))

tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X.iloc[train_idx], y[train_idx])
print(export_text(tree, feature_names=features))

## 4. Results: model vs baseline on the same unseen clients

In [ ]:
y_test = y[test_idx]
tree_score = tree.predict_proba(X.iloc[test_idx])[:, 1]
rule_score = df["hand_rule_score"].values[test_idx]

print("Test pages:", len(test_idx), "| base rate (declining):", round(y_test.mean(), 3))
results = pd.DataFrame({
    "Precision@20": [precision_at_k(rule_score, y_test, 20), precision_at_k(tree_score, y_test, 20)],
    "Precision@50": [precision_at_k(rule_score, y_test, 50), precision_at_k(tree_score, y_test, 50)],
}, index=["Hand rule", "Decision tree (depth 3)"]).round(3)
results

## 5. Leakage check (demonstration only)
The label comes from the trend fields. Adding `trend_pct` as an input produces a "perfect" score that means nothing.

In [ ]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print("Leaky tree Precision@50 (scored on its own training data):",
      round(precision_at_k(leaky.predict_proba(X_leaky)[:, 1], y, 50), 3), "<- leakage, not skill")

## 6. Limitations
One random split and one seed, so numbers could shift on another split. Precision@20 rests on only 20 pages. "Declining" is a simple trend label, not a cause. This is a small anonymized slice. Nothing here shows that refreshing a page improves it, and nothing predicts how Google's algorithm works.

## 7. Ranked recommendations (decision support)
1. **Review first:** stale (>= 180 days) and visible (>= 500 impressions in 90 days) pages, ordered by impressions.
2. **Monitor next:** visible pages that were updated more recently.
3. **Low priority for now:** pages with little visibility; revisit if impressions rise.
4. **Before relying on this at scale:** repeat on the full release with several random splits.

## 8. Reproducibility
Seed 42 throughout. Repo: https://github.com/karyllegt/ml-starter-pack-flyrank-ai

## 9. Acknowledgments and data credit
Built on the FlyRank ML Internship dataset (https://flyrank.ai).